# 03 — Visualizing the KG

Quick, fun proof that `kg/vienna_mobility_kg.ttl` actually works — not a deep
analysis, just a look at what's in there. Three parts:
1. A bar chart of what's actually in the graph
2. A scatter "map" of every POI, colored by type — no basemap, just raw
   coordinates, and Vienna's shape shows up anyway
3. Two zoomed-in mini graph diagrams showing the actual node-link structure
   for one Park and one transit Stop

Plus a small bonus at the end: a taste of the proximity reasoning the KG makes
possible — "what's the nearest playground to the Albertina?"

Only uses libraries already in `pyproject.toml` (rdflib, pandas, matplotlib) —
no new dependencies needed.

## Load the graph

In [ ]:
from rdflib import Graph
import pandas as pd
import matplotlib.pyplot as plt
import math

g = Graph()
g.parse("../kg/vienna_mobility_kg.ttl", format="turtle")
print(f"Loaded {len(g)} triples")

## 1. What's actually in the graph

One SPARQL query, one bar chart.

In [ ]:
q = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
SELECT ?class (COUNT(?x) AS ?n) WHERE {
    ?x a ?class .
    FILTER(?class IN (
        schema:Museum, schema:Library, viennakg:BathingSite, schema:Park,
        viennakg:SwimmingPool, viennakg:PlaygroundArea, schema:TouristAttraction,
        viennakg:Stop, viennakg:Platform, viennakg:Line
    ))
}
GROUP BY ?class
ORDER BY DESC(?n)
"""
counts = {g.qname(row["class"]).split(":")[-1]: int(row.n) for row in g.query(q)}

fig, ax = plt.subplots(figsize=(9, 5))
names = list(counts.keys())
values = list(counts.values())
ax.barh(names, values, color="#4C72B0")
ax.set_xlabel("instances")
ax.set_title("Vienna Mobility KG — instances per class")
ax.invert_yaxis()
for i, v in enumerate(values):
    ax.text(v, i, f"  {v:,}", va="center")
plt.tight_layout()
plt.show()

## 2. A "map" made entirely of KG coordinates

No basemap, no tiles — just every POI's `geo:lat`/`geo:long` plotted as a
scatter point, colored by class. If the modelling worked, Vienna's outline
should show up on its own.

In [ ]:
POI_CLASSES = {
    "Museum": "schema:Museum",
    "Library": "schema:Library",
    "BathingSite": "viennakg:BathingSite",
    "Park": "schema:Park",
    "SwimmingPool": "viennakg:SwimmingPool",
    "PlaygroundArea": "viennakg:PlaygroundArea",
    "TouristAttraction": "schema:TouristAttraction",
}

def fetch_coords(qname):
    q = f"""
    PREFIX schema: <https://schema.org/>
    PREFIX viennakg: <http://example.org/viennakg#>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    SELECT ?lon ?lat WHERE {{
        ?poi a {qname} ; geo:long ?lon ; geo:lat ?lat .
    }}
    """
    return [(float(r.lon), float(r.lat)) for r in g.query(q)]

# Stops in light gray as geographic context (there are a lot of them, small + faint)
stop_coords = fetch_coords("viennakg:Stop")
stop_lon, stop_lat = zip(*stop_coords)

fig, ax = plt.subplots(figsize=(9, 10))
ax.scatter(stop_lon, stop_lat, s=2, color="lightgray", label=f"Wiener Linien stops ({len(stop_coords):,})")

for label, qname in POI_CLASSES.items():
    coords = fetch_coords(qname)
    lon, lat = zip(*coords)
    ax.scatter(lon, lat, s=8, alpha=0.7, label=f"{label} ({len(coords):,})")

ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Every POI + transit stop in the KG, plotted by coordinates alone")
ax.set_aspect("equal")
ax.legend(markerscale=3, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 3. Zooming into the actual graph structure

The map above proves the *data* is right, but a knowledge graph is about the
*links*, not just points on a map. Two small, manually-laid-out node-link
diagrams — small enough to draw by hand with plain matplotlib, no graph
library needed.

### 3a. A Park and its amenity features

Every `schema:Park` connects to three `schema:LocationFeatureSpecification`
nodes via `schema:amenityFeature` — this is what that actually looks like for
one park.

In [ ]:
def draw_star(center_node, leaf_nodes, leaf_colors=None, title=""):
    """Small helper: one center node, its leaves arranged in a circle around it."""
    n = len(leaf_nodes)
    fig, ax = plt.subplots(figsize=(6, 6))
    cx, cy = 0, 0
    ax.scatter([cx], [cy], s=1800, color="#4C72B0", zorder=3)
    ax.text(cx, cy, center_node, ha="center", va="center", color="white", fontsize=9, weight="bold", zorder=4)

    for i, leaf in enumerate(leaf_nodes):
        angle = 2 * math.pi * i / n
        x, y = 2.4 * math.cos(angle), 2.4 * math.sin(angle)
        color = leaf_colors[i] if leaf_colors else "#DD8452"
        ax.plot([cx, x], [cy, y], color="gray", zorder=1, linewidth=1)
        ax.scatter([x], [y], s=1400, color=color, zorder=3)
        ax.text(x, y, leaf, ha="center", va="center", fontsize=8, wrap=True, zorder=4)

    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

q = """
PREFIX schema: <https://schema.org/>
SELECT ?fname ?fvalue WHERE {
    ?p a schema:Park ; schema:name "Stadtpark" ; schema:amenityFeature ?f .
    ?f schema:name ?fname ; schema:value ?fvalue .
}
"""
leaves, colors = [], []
for row in g.query(q):
    val = row.fvalue.toPython()
    leaves.append(f"{row.fname}\n({'yes' if val else 'no'})")
    colors.append("#55A868" if val else "#C44E52")

draw_star("Stadtpark", leaves, colors, "Stadtpark —schema:amenityFeature→ LocationFeatureSpecification")

### 3b. A transit Stop, its Platforms, and the Lines serving them

Structurally different from the Park example above — a two-hop chain
(`Stop —hasPlatform→ Platform —servedByLine→ Line`), and this particular stop
shows the "shared RBL across directions" pattern discussed earlier.

In [ ]:
stop_name = "Aderklaaer Straße"

q = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
SELECT ?rbl ?lineName WHERE {
    ?stop a viennakg:Stop ; schema:name "%s" ; viennakg:hasPlatform ?platform .
    ?platform viennakg:rbl ?rbl ; viennakg:servedByLine ?line .
    ?line schema:name ?lineName .
}
ORDER BY ?rbl
""" % stop_name

fig, ax = plt.subplots(figsize=(8, 6))
cx, cy = 0, 2
ax.scatter([cx], [cy], s=2000, color="#4C72B0", zorder=3)
ax.text(cx, cy, stop_name, ha="center", va="center", color="white", fontsize=8, weight="bold", zorder=4)

rows = list(g.query(q))
n = len(rows)
for i, row in enumerate(rows):
    x = (i - (n - 1) / 2) * 2.2
    y_platform, y_line = 0, -2
    ax.plot([cx, x], [cy, y_platform], color="gray", linewidth=1, zorder=1)
    ax.scatter([x], [y_platform], s=1400, color="#DD8452", zorder=3)
    ax.text(x, y_platform, f"RBL\n{row.rbl}", ha="center", va="center", fontsize=8, zorder=4)
    ax.plot([x, x], [y_platform, y_line], color="gray", linewidth=1, zorder=1)
    ax.scatter([x], [y_line], s=1400, color="#55A868", zorder=3)
    ax.text(x, y_line, str(row.lineName), ha="center", va="center", fontsize=9, weight="bold", zorder=4)

ax.set_xlim(-n * 1.3, n * 1.3)
ax.set_ylim(-3, 3.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"{stop_name} —hasPlatform→ Platform —servedByLine→ Line")
plt.tight_layout()
plt.show()

## Bonus: a taste of proximity reasoning

Nothing fancy — just haversine distance over `geo:lat`/`geo:long`, the same
coordinates plotted above. This is a preview of what the Reasoning Layer will
do properly; here it's just "which playground is closest to the Albertina?".

In [ ]:
def haversine_m(lon1, lat1, lon2, lat2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * R * math.asin(math.sqrt(a))

q_museum = """
PREFIX schema: <https://schema.org/>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?lon ?lat WHERE {
    ?m a schema:Museum ; schema:name "Albertina" ; geo:long ?lon ; geo:lat ?lat .
}
"""
m_lon, m_lat = [(float(r.lon), float(r.lat)) for r in g.query(q_museum)][0]

q_playgrounds = """
PREFIX schema: <https://schema.org/>
PREFIX viennakg: <http://example.org/viennakg#>
PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
SELECT ?name ?lon ?lat WHERE {
    ?p a viennakg:PlaygroundArea ; schema:name ?name ; geo:long ?lon ; geo:lat ?lat .
}
"""
nearest = min(
    ((row.name, float(row.lon), float(row.lat),
      haversine_m(m_lon, m_lat, float(row.lon), float(row.lat)))
     for row in g.query(q_playgrounds)),
    key=lambda x: x[3],
)
print(f"Nearest playground to the Albertina: {nearest[0]}, ~{nearest[3]:.0f} m away")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter([m_lon], [m_lat], s=200, color="#4C72B0", label="Albertina", zorder=3)
ax.scatter([nearest[1]], [nearest[2]], s=200, color="#55A868", label=nearest[0], zorder=3)
ax.plot([m_lon, nearest[1]], [m_lat, nearest[2]], "--", color="gray", zorder=1)
ax.set_title(f"~{nearest[3]:.0f} m apart")
ax.set_aspect("equal")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Wrap-up

All three views (counts, map, structure) pull from the exact same
`kg/vienna_mobility_kg.ttl` file `kg/ingestion/build_kg.py` produces — nothing
here is hand-faked. If any of these looked broken (empty map, disconnected
graph, wrong counts), that'd be a real signal something in the ingestion is
off. Since they all check out, the KG is doing its job.